In [0]:
# import important libraries
from pyspark.sql.functions import *

In [0]:
storage_account_name = "polishjobpipelinebronze"

container_name = "bronze"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net", storage_account_access_key)


In [0]:
path = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net/justjoin/"
dbutils.fs.ls(path)


[FileInfo(path='wasbs://bronze@polishjobpipelinebronze.blob.core.windows.net/justjoin/2026-02-23/', name='2026-02-23/', size=0, modificationTime=0)]

In [0]:
# load json files from path
path = "wasbs://bronze@polishjobpipelinebronze.blob.core.windows.net/justjoin/2026-02-23/"

df = spark.read.option("multiline", "true").json(path)
df.show(5)


+---------+--------------------+---------+--------------------+------------------+--------+--------------------+--------------------+---------------------------+------------------------------+------------------+-----------+--------------------+-----------+-------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+--------+----------------------+----------+----------+--------------------+--------------------+----------+----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------+-----------+-------------+
|appliedAt|            applyUrl|bannerUrl|                body|          category|    city|      companyLogoUrl|         companyName|companyProfileCoverPhotoUrl|companyProfileShortDescription|companyProfileSlug|companySize|          companyUrl|countryCode|customConsent|     employmentTypes|experienceLevel|           expiredAt|       futureCon

In [0]:
# print the schema
df.printSchema()

root
 |-- appliedAt: string (nullable = true)
 |-- applyUrl: string (nullable = true)
 |-- bannerUrl: string (nullable = true)
 |-- body: string (nullable = true)
 |-- category: struct (nullable = true)
 |    |-- key: string (nullable = true)
 |    |-- parentKey: string (nullable = true)
 |-- city: string (nullable = true)
 |-- companyLogoUrl: string (nullable = true)
 |-- companyName: string (nullable = true)
 |-- companyProfileCoverPhotoUrl: string (nullable = true)
 |-- companyProfileShortDescription: string (nullable = true)
 |-- companyProfileSlug: string (nullable = true)
 |-- companySize: string (nullable = true)
 |-- companyUrl: string (nullable = true)
 |-- countryCode: string (nullable = true)
 |-- customConsent: string (nullable = true)
 |-- employmentTypes: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- currency: string (nullable = true)
 |    |    |-- currencySource: string (nullable = true)
 |    |    |-- from: double (nullable = tr

In [0]:
# select the important colums
df_core = df.select(
    "id",
    "slug",
    "title",
    "city",
    "street",
    "companyName",
    "companySize",
    "countryCode",
    "experienceLevel",
    "workplaceType",
    "workingTime",
    "publishedAt",
    "isActive",
    "category.key",
    "requiredSkills",
    "employmentTypes",
    "locations",
    "languages"
)
df_core

DataFrame[id: string, slug: string, title: string, city: string, street: string, companyName: string, companySize: string, countryCode: string, experienceLevel: string, workplaceType: string, workingTime: string, publishedAt: string, isActive: boolean, key: string, requiredSkills: array<struct<level:bigint,name:string>>, employmentTypes: array<struct<currency:string,currencySource:string,from:double,fromPerUnit:double,gross:boolean,to:double,toPerUnit:double,type:string,unit:string>>, locations: array<struct<city:string,latitude:double,longitude:double,slug:string,street:string>>, languages: array<struct<code:string,level:string>>]

In [0]:
# change the key to category 
df_core = df_core.withColumnRenamed("key", "category")
df_core.select("companyName", "category", "city", "publishedAt").show(5)

+--------------------+----------+--------+--------------------+
|         companyName|  category|    city|         publishedAt|
+--------------------+----------+--------+--------------------+
|           VirtusLab|      java|  Kraków|2026-02-22T05:00:...|
|STATSCORE Sp. z o.o.|javascript|Katowice|2026-02-19T14:00:...|
|STATSCORE Sp. z o.o.|       php|Katowice|2026-02-19T11:00:...|
|ERGO Technology &...|        pm|Warszawa|2026-02-18T10:00:...|
|ERGO Technology &...|    python|Warszawa|2026-02-20T13:00:...|
+--------------------+----------+--------+--------------------+
only showing top 5 rows



In [0]:
# change the publishedAt from string to timestamp
df_core = df_core.withColumn("publishedAt", to_timestamp(col('publishedAt')))

In [0]:
df_core.select("requiredSkills").show(2)

+--------------------+
|      requiredSkills|
+--------------------+
|[{1, Kotlin Corou...|
|[{2, Nest.js}, {4...|
+--------------------+
only showing top 2 rows



In [0]:

# Create the silver_skills
df_skills = df_core.select(
    "id", 
    "title", 
    "city", 
    "companyName", 
    "experienceLevel",
    explode("requiredSkills").alias("skill")
).select(
    "id",
    "title",
    "city",
    "companyName",
    "experienceLevel",
    col("skill.name").alias("skill_name"),
    col("skill.level").alias("skill_level")
)

In [0]:
df_salaries = df_core.select(
    "id",
    "title",
    "city",
    "companyName",
    "experienceLevel",
    explode("employmentTypes").alias("salary")
).select(
    "id",
    "title",
    "city",
    "companyName",
    "experienceLevel",
    col("salary.type").alias("contract_type"),
    col("salary.from").alias("salary_min"),
    col("salary.to").alias("salary_max"),
    col("salary.currency").alias("currency"),
    col("salary.unit").alias("salary_unit"),
    col("salary.gross").alias("is_gross"),
    col("salary.currencySource").alias("salary_currencySource")
)

In [0]:
df_salaries.show(2)

+--------------------+--------------------+------+-----------+---------------+-------------+----------+----------+--------+-----------+--------+---------------------+
|                  id|               title|  city|companyName|experienceLevel|contract_type|salary_min|salary_max|currency|salary_unit|is_gross|salary_currencySource|
+--------------------+--------------------+------+-----------+---------------+-------------+----------+----------+--------+-----------+--------+---------------------+
|079c8cd2-8ab5-408...|IDE Engineer (Sen...|Kraków|  VirtusLab|         senior|          b2b|    4622.0|    5943.0|     CHF|      Month|   false|           conversion|
|079c8cd2-8ab5-408...|IDE Engineer (Sen...|Kraków|  VirtusLab|         senior|          b2b|    4148.0|    5333.0|     GBP|      Month|   false|           conversion|
+--------------------+--------------------+------+-----------+---------------+-------------+----------+----------+--------+-----------+--------+---------------------

In [0]:
# filter out other currency and keep the original value(PLN)
df_salaries_clean = df_salaries.filter(df_salaries['salary_currencySource'] == 'original')

In [0]:
# convert the salary_unit to lowercase to standidize 
df_salaries_clean = df_salaries_clean.withColumn("salary_unit", lower("salary_unit"))

In [0]:
# create a new column with salary_disclosed flag

df_salaries_clean = df_salaries_clean.withColumn(
    "salary_disclosed",
    when(col("salary_min").isNull(), False).otherwise(True)
)

In [0]:
df_core.show(2)

+--------------------+--------------------+--------------------+--------+---------+--------------------+-----------+-----------+---------------+-------------+-----------+--------------------+--------+----------+--------------------+--------------------+--------------------+----------+
|                  id|                slug|               title|    city|   street|         companyName|companySize|countryCode|experienceLevel|workplaceType|workingTime|         publishedAt|isActive|  category|      requiredSkills|     employmentTypes|           locations| languages|
+--------------------+--------------------+--------------------+--------+---------+--------------------+-----------+-----------+---------------+-------------+-----------+--------------------+--------+----------+--------------------+--------------------+--------------------+----------+
|079c8cd2-8ab5-408...|virtuslab-ide-eng...|IDE Engineer (Sen...|  Kraków| Szlak 49|           VirtusLab|    101-500|         PL|         senio

In [0]:
# write to file 
silver_path = f"wasbs://silver@{storage_account_name}.blob.core.windows.net"

# write skills table
(df_skills.write
        .format('delta')
        .mode("overwrite")
        .save(f"{silver_path}/justjoin/skills"))

# write salary table
(df_salaries_clean.drop("salary_currencySource")
                .write
                .format("delta")
                .mode("overwrite")
                .save(f"{silver_path}/justjoin/salaries")
)

# write core table
(df_core
        .drop('employmentTypes', "locations", "requiredSkills", "languages")
        .write
        .format("delta")
        .mode("overwrite")
        .save(f"{silver_path}/justjoin/jobs"))



In [0]:
job_count = spark.read.format("delta").load(f"{silver_path}/justjoin/jobs").count()
skill_count = spark.read.format("delta").load(f"{silver_path}/justjoin/skills").count()
salaries_count = spark.read.format("delta").load(f"{silver_path}/justjoin/salaries").count()

print(f"Job Count: {job_count}")
print(f"Number of skill: {skill_count}")
print(f"Salaries Count: {salaries_count}")

Job Count: 946
Number of skill: 5184
Salaries Count: 1095


In [0]:
spark.read.format("delta").load(f"{silver_path}/justjoin/jobs").printSchema()
spark.read.format("delta").load(f"{silver_path}/justjoin/skills").printSchema()
spark.read.format("delta").load(f"{silver_path}/justjoin/salaries").printSchema()

root
 |-- id: string (nullable = true)
 |-- slug: string (nullable = true)
 |-- title: string (nullable = true)
 |-- city: string (nullable = true)
 |-- street: string (nullable = true)
 |-- companyName: string (nullable = true)
 |-- companySize: string (nullable = true)
 |-- countryCode: string (nullable = true)
 |-- experienceLevel: string (nullable = true)
 |-- workplaceType: string (nullable = true)
 |-- workingTime: string (nullable = true)
 |-- publishedAt: timestamp (nullable = true)
 |-- isActive: boolean (nullable = true)
 |-- category: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- city: string (nullable = true)
 |-- companyName: string (nullable = true)
 |-- experienceLevel: string (nullable = true)
 |-- skill_name: string (nullable = true)
 |-- skill_level: long (nullable = true)

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- city: string (nullable = true)
 |-- companyName: string